# Repaso

## Montar el cluster

**Historial de comandos:**

docker-compose -f .\docker-compose.yml up -d

**Conectarse a configsvr**

docker exec -it configsvr mongosh --port 27019

**Y ejecutar:**

rs.initiate({_id: "configReplSet", configsvr: true, members: [{_id: 0, host: "configsvr:27019"}]})

**Conectarse a mongodb1**

docker exec -it mongodb1 mongosh --port 27017

**Y ejecutar:**

rs.initiate({_id: "rs0", members: [{_id: 0, host: "mongodb1:27017"}, {_id: 1, host: "mongodb2:27017"}, {_id: 2, host: "mongodb3:27017"}]})

**Conectarse a mongosh**

docker exec -it mongos mongosh --port 27017

**Y ejecutar:**

sh.addShard("rs0/mongodb1:27017,mongodb2:27017,mongodb3:27017")

**Verificar que todo va bien con sh.status()**

## Ejercicio 1

Una vez con el cluster montado, ejecutamos:

In [1]:
from pymongo import MongoClient

#Utiliza la IP de tu anfitrión
cliente = MongoClient('mongos', 27017)

# comprobamos que es mongos
if cliente.admin.command("hello").get("msg") == "isdbgrid":
    print("OK: conectado a mongos")


#Creamos la instancia para interactuar con la colección de clientes y pedidos
bbdd = cliente.tienda
coleccion_clientes = bbdd.clientes
coleccion_pedidos = bbdd.pedidos
coleccion_productos = bbdd.productos

#shardeamos si es necesario si no pass

try:
    cliente.admin.command({
        "shardCollection": "tienda.clientes",
        "key": {"dni": 1}
    })
    cliente.admin.command({
        "shardCollection": "tienda.pedidos",
        "key": {"dni": 1}
    })
    cliente.admin.command({
        "shardCollection": "tienda.productos",
        "key": {"dni": 1}
    })
except:
    pass

#ANTES APLIQUEMOS ENABLE SHARDING A LA BBDD Y A LA COLECCIÓN

OK: conectado a mongos


In [12]:
#Inserción de documentos de clientes
documentos_clientes = [
    {'IDCliente': 1, 'Nombre': "Juan", 'Apellidos': "Fernández"},
    {'IDCliente': 2, 'Nombre': "María", 'Apellidos': "Fernández"},
    {'IDCliente': 3, 'Nombre': "Carolina", 'Apellidos': "Pérez"}
]

resultado = coleccion_clientes.insert_many(documentos_clientes)
print("Se han insertado",len(resultado.inserted_ids),"clientes")

Se han insertado 3 clientes


In [3]:
#Inserción de documentos de pedidos
documentos_pedidos = [
    {'IDPedido': 1, 'IDCliente': 1, 'Importe': 3.24, 'Ciudad': "Vigo"},
    {'IDPedido': 2, 'IDCliente': 1, 'Importe': 8.01, 'Ciudad': "Pontevedra"},
    {'IDPedido': 3, 'IDCliente': 3, 'Importe': 28.12, 'Ciudad': "A Coruña"},
    {'IDPedido': 4, 'IDCliente': 1, 'Importe': 56.78, 'Ciudad': "Vigo"},
    {'IDPedido': 5, 'IDCliente': 2, 'Importe': 0.12, 'Ciudad': "Madrid"},
    {'IDPedido': 6, 'IDCliente': 3, 'Importe': 99.45, 'Ciudad': "Barcelona"},
    {'IDPedido': 7, 'IDCliente': 3, 'Importe': 2.1, 'Ciudad': "Valencia"},
    {'IDPedido': 8, 'IDCliente': 1, 'Importe': 9, 'Ciudad': "Ourense"},
    {'IDPedido': 9, 'IDCliente': 1, 'Importe': 32.56, 'Ciudad': "Lugo"},
    {'IDPedido': 10, 'IDCliente': 3, 'Importe': 5.45, 'Ciudad': "Santiago"},
]

resultado = coleccion_pedidos.insert_many(documentos_pedidos)
print("Se han insertado",len(resultado.inserted_ids),"pedidos")

Se han insertado 10 pedidos


**Entra en la shell de mongos:**

docker exec -it mongos mongosh --port 27017

**Cambiamos a nuestra base de datos**
   
use tienda

**Habilitamos sharding**

sh.enableSharding("tienda")

**Shardeamos las colecciones usando el _id como clave**
   
sh.shardCollection("tienda.productos", { "_id": 1})

sh.shardCollection("tienda.clientes", { "_id": 1})

sh.shardCollection("tienda.pedidos", {"_id": 1})

In [14]:
import pandas as pd

try:
    # Asegúrate de que el archivo productos.csv esté en la misma carpeta que el notebook
    df = pd.read_csv('productos.csv')
    
    # Convertimos el DataFrame a una lista de diccionarios para MongoDB
    datos = df.to_dict(orient='records')
    print(f"Archivo leído: {len(datos)} registros listos para importar.")
    
    # 3. Importación masiva
    resultado = coleccion_productos.insert_many(datos)
    print(f"Éxito: Se han insertado {len(resultado.inserted_ids)} documentos.")
    
except FileNotFoundError:
    print("Error: No se encontró el archivo 'productos.csv'.")
except Exception as e:
    print(f"Error durante la importación: {e}")

# 4. Comprobación de los primeros 10 documentos
print("\n--- MUESTRA DE LOS PRIMEROS 10 DOCUMENTOS ---")
cursor = coleccion_productos.find().limit(10)
for i, doc in enumerate(cursor, 1):
    print(f"{i}: {doc}")

Archivo leído: 20 registros listos para importar.
Éxito: Se han insertado 20 documentos.

--- MUESTRA DE LOS PRIMEROS 10 DOCUMENTOS ---
1: {'_id': ObjectId('697a042b62cbdb93388e7da2'), 'nombre': 'Portátil Pro 14', 'precio': 89.99, 'stock': 17, 'familia': 'informatica'}
2: {'_id': ObjectId('697a042b62cbdb93388e7da3'), 'nombre': 'Ratón Óptico USB', 'precio': 14.95, 'stock': 125, 'familia': 'informatica'}
3: {'_id': ObjectId('697a042b62cbdb93388e7da4'), 'nombre': 'Teclado Mecánico', 'precio': 79.9, 'stock': 50, 'familia': 'informatica'}
4: {'_id': ObjectId('697a042b62cbdb93388e7da5'), 'nombre': 'Monitor 27"', 'precio': 229.99, 'stock': 23, 'familia': 'informatica'}
5: {'_id': ObjectId('697a042b62cbdb93388e7da6'), 'nombre': 'SSD 1TB', 'precio': 119.0, 'stock': 40, 'familia': 'informatica'}
6: {'_id': ObjectId('697a042b62cbdb93388e7da7'), 'nombre': 'Impresora Láser', 'precio': 219.98900000000003, 'stock': 10, 'familia': 'oficina'}
7: {'_id': ObjectId('697a042b62cbdb93388e7da8'), 'nombre': '

**Entrar en la mongoshell y ejecutar:**

**Entrar al shell**

docker exec -it mongos mongosh --port 27017

**Comandos dentro de mongosh:**

use tienda

show collections

db.productos.countDocuments()

![SimulacroExamen01](./img/SimulacroExamen01.png)

## Ejercicio 2

**Asegurar que la base de datos permite sharding**

sh.enableSharding("tienda")

**Crear el índice necesario para la shard key (Hashed)**

use tienda

db.productos.createIndex({ "_id": "hashed" })

**Shardear la colección usando el _id**

sh.shardCollection("tienda.productos", { "_id": 1 })

![SimulacroExamen02](./img/SimulacroExamen02.png)

In [13]:
# Consulta a la base de datos 'config' que es donde mongos guarda la info del sharding
db_config = cliente['config']
coleccion_shards = db_config['collections']

# Buscamos la info de nuestra colección
info_sharding = coleccion_shards.find_one({"_id": "tienda.productos"})

if info_sharding:
    print("Confirmación de Sharding:")
    print(f"ID Colección: {info_sharding['_id']}")
    print(f"Shard Key: {info_sharding['key']}")
    print(f"Estado: Shardeada")
else:
    print("No se encontró información de sharding. Revisa si ejecutaste sh.shardCollection()")

No se encontró información de sharding. Revisa si ejecutaste sh.shardCollection()


## Ejercicio 3

In [6]:
# 1. Productos con stock menor que 10
print("--- PRODUCTOS CON POCO STOCK (< 10) ---")
bajo_stock = coleccion_productos.find({"stock": {"$lt": 10}})
for p in bajo_stock:
    print(f"Producto: {p['nombre']} - Stock: {p['stock']}")

# 2. Productos de la familia 'electronica'
print("\n--- PRODUCTOS DE INFORMÁTICA ---")
electronica = coleccion_productos.find({"familia": "informatica"})
for p in electronica:
    print(f"Producto: {p['nombre']} - Familia: {p['familia']}")

# 3. Los 3 productos más caros
# Sort -1 indica orden descendente
print("\n--- TOP 3 PRODUCTOS MÁS CAROS ---")
mas_caros = coleccion_productos.find().sort("precio", -1).limit(3)
for p in mas_caros:
    print(f"Producto: {p['nombre']} - Precio: {p['precio']}€")

--- PRODUCTOS CON POCO STOCK (< 10) ---
Producto: Mesa Oficina - Stock: 8
Producto: Cafetera Express - Stock: 9
Producto: Robot Cocina - Stock: 6
Producto: Bicicleta Montaña - Stock: 5

--- PRODUCTOS DE INFORMÁTICA ---
Producto: Portátil Pro 14 - Familia: informatica
Producto: Ratón Óptico USB - Familia: informatica
Producto: Teclado Mecánico - Familia: informatica
Producto: Monitor 27" - Familia: informatica
Producto: SSD 1TB - Familia: informatica

--- TOP 3 PRODUCTOS MÁS CAROS ---
Producto: Portátil Pro 14 - Precio: 899.99€
Producto: Bicicleta Montaña - Precio: 599.0€
Producto: Robot Cocina - Precio: 399.0€


## Ejercicio 4

In [7]:
# 1. Subir un 10% los precios de la familia 'oficina'
# Usamos $mul para multiplicar el valor actual por 1.10
res_oficina = coleccion_productos.update_many(
    {"familia": "oficina"},
    {"$mul": {"precio": 1.10}}
)
print(f"Precios actualizados en oficina: {res_oficina.modified_count} productos.")

# 2. Incrementar stock en +5 para 'informatica'
# Usamos $inc para sumar una cantidad al valor actual
res_informatica = coleccion_productos.update_many(
    {"familia": "informatica"},
    {"$inc": {"stock": 5}}
)
print(f"Stock incrementado en informática: {res_informatica.modified_count} productos.")

Precios actualizados en oficina: 5 productos.
Stock incrementado en informática: 5 productos.


In [8]:
# 3. Cambiar el precio de un producto concreto (por su nombre o ID)
# Ajusta el nombre según los datos de tu CSV
res_unico = coleccion_productos.update_one(
    {"nombre": "Portátil Pro 14"}, 
    {"$set": {"precio": 89.99}}
)
print(f"Producto concreto actualizado: {res_unico.modified_count}")

# 4. Eliminar productos con stock igual a 0
res_eliminados = coleccion_productos.delete_many({"stock": 0})
print(f"Productos eliminados (stock 0): {res_eliminados.deleted_count}")

Producto concreto actualizado: 1
Productos eliminados (stock 0): 0


## Ejercicio 5

In [9]:
# 1. Índice simple sobre 'familia'
# 1 significa orden ascendente
nombre_indice_simple = coleccion_productos.create_index([("familia", 1)])
print(f"Índice simple creado: {nombre_indice_simple}")

# 2. Índice compuesto sobre 'familia' (asc) y 'precio' (desc)
# El orden en el precio (-1) es útil para buscar "los más caros de la familia X"
nombre_indice_compuesto = coleccion_productos.create_index([("familia", 1), ("precio", -1)])
print(f"Índice compuesto creado: {nombre_indice_compuesto}")

# Listar todos los índices para la captura de pantalla
print("\n--- Índices actuales en la colección ---")
for index in coleccion_productos.list_indexes():
    print(index)

Índice simple creado: familia_1
Índice compuesto creado: familia_1_precio_-1

--- Índices actuales en la colección ---
SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])
SON([('v', 2), ('key', SON([('familia', 1)])), ('name', 'familia_1')])
SON([('v', 2), ('key', SON([('familia', 1), ('precio', -1)])), ('name', 'familia_1_precio_-1')])


In [10]:
# 3. Usar explain() en una consulta que use los campos indexados
# Queremos ver cómo busca productos de 'electronica' ordenados por precio
explicacion = coleccion_productos.find({"familia": "electronica"}).sort("precio", -1).explain()

# Mostramos la parte del plan de ejecución
import pprint
print("--- PLAN DE EJECUCIÓN (Winning Plan) ---")
pprint.pprint(explicacion['queryPlanner']['winningPlan'])

--- PLAN DE EJECUCIÓN (Winning Plan) ---
{'shards': [{'connectionString': 'rs0/mongodb1:27018,mongodb2:27018,mongodb3:27018',
             'indexFilterSet': False,
             'maxIndexedAndSolutionsReached': False,
             'maxIndexedOrSolutionsReached': False,
             'maxScansToExplodeReached': False,
             'namespace': 'tienda.productos',
             'parsedQuery': {'familia': {'$eq': 'electronica'}},
             'rejectedPlans': [{'inputStage': {'inputStage': {'direction': 'forward',
                                                              'indexBounds': {'familia': ['["electronica", '
                                                                                          '"electronica"]']},
                                                              'indexName': 'familia_1',
                                                              'indexVersion': 2,
                                                              'isMultiKey': False,
               

## Ejercicio 6

**¿Por qué las aplicaciones se conectan a mongo?**

Ya que mongos sirve de intérprete o enlace entre esas aplicaciones y los nodos de datos donde está almacenada la información. Mongos se encarga de la conexión, consulta y autentificación(si hay), ya que si cada aplicación tuviera que conectarse individualmente a cada nodo de datos requerido, sería todo mucho más caótico.

**¿Qué función tienen los config servers?**

Sirven para guardar los metadatos de la información, es decir, como su nombre indica, su configuración, localización, estado, en que shard se encuentra...

| Componente         | Función Principal                                                              |
|--------------------|--------------------------------------------------------------------------------|
| **Mongos**         | Enrutador. Dirige las consultas al shard correcto.                             |
| **Config Servers** | Metadatos. Guardan el mapa de distribución de los datos.                       |
| **Replica Set**    | Disponibilidad. Garantiza que los datos sigan accesibles si falla un servidor. |


**¿Qué ocurre si cae un nodo del replicaset?**

Depende, si es uno secundario, en cuanto el cluster deja de notar sus heartbeats, simplemente deja de usarlo y la información que contiene queda congelada hasta que ese nodo vuelva a estar operativo. El cluster sigue funcionando, aunque dependiendo de su tamaño, se producirá pérdida de redundancia.

Si es un nodo primario el que cae, en cuanto el cluster lo detecta, los nodos secundarios establecen otro primario mediante votación, y ese nodo pasa a ocuparse del trabajo del anterior.

Como es un replicaset, nunca hay pérdida de la información, ya que esta está alojada en todos los nodos.

| Si cae el... | Impacto inmediato      | Acción del Clúster                                               |
|--------------|------------------------|------------------------------------------------------------------|
| Primario     | Pausa breve (segundos) | Elección de un nuevo Primario.                                   |
| Secundario   | Ninguno                | El Primario sigue solo; el secundario se sincroniza al volver.   |